In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

import gurobipy as grb

In [79]:
portfolio_1 = [
    {
        'type': 'stock', # call / put / future / stock
        'quantity': 10,
        'daily_limit': 10000,
        'execution_lag': 1,
        'ds': True,
        'days_to_maturity': None
    },
    {
        'type': 'future', # call / put / future / stock
        'strike': 90,
        'quantity': -100,
        'daily_limit': 10000,
        'execution_lag': 10,
        'ds': True,
        'days_to_maturity': 360
    },
]

In [80]:
def get_liquidation_period(portfolio):
    check = [i['execution_lag'] + math.ceil(abs(i['quantity']) / i['daily_limit']) for i in portfolio]

    return max(check) - 1

T = get_liquidation_period(portfolio_1) # в днях
T

10

In [85]:
spot = 100
corr = 0.99
mu = np.array([0.01, 0.01])
sigma = np.array([0.2, 0.2])

r = 0.01

N = 1000

In [86]:
def future_price(S0, r, q, T):
    return S0 * np.exp((r - q) * T)

def value_of_future(F0, K, r, T):
    return (F0 - K)* np.exp(-r*T)

In [87]:
def gbm_simulate(seed, S0, mu, sigma, cov, T):
    '''

    :param seed: seed
    :param S0: spot price at t=0
    :param mu: drift
    :param sigma: volatility
    :param cov: covamatrix
    :param T: number of days to simulate
    :return:
    '''
    np.random.seed(seed)
    dim = np.size(S0)

    t = np.arange(0, T+1) * 1/252
    A = np.linalg.cholesky(cov)
    S = np.zeros([dim, len(t)])
    S[:, 0] = S0

    for i in range(1, len(t)):
        drift = (mu - sigma**2 / 2) * (t[i] - t[i-1])
        W = np.random.normal(0.0, 1.0, dim)
        diffusion = np.matmul(A,W) * np.sqrt(t[i] - t[i-1])

        S[:, i] = S[:, i-1] * np.exp(drift + diffusion)

    return S, t

In [88]:
gbm_simulate(22,
             spot, mu, sigma,
             np.array([[sigma[0], sigma[0]*sigma[1]],
                       [sigma[0]*sigma[1], sigma[1]]]),
             T)[0]

LinAlgError: Matrix is not positive definite

In [65]:
SS = []
for i in range(N):
    seed = int(np.random.uniform(1, 2**32-1, 1))
    SS.append(gbm_simulate(seed,spot, mu, sigma,
             np.array([[sigma[0], sigma[0]*sigma[1]],
                       [sigma[0]*sigma[1], sigma[1]]]),
             T)[0])

In [66]:
price_paths = np.array(SS)
price_paths[:, 1, :] = future_price(price_paths[:, 1, :], r, 0, np.arange(0, T+1) * 1/252)
price_paths[:, 1, :] = value_of_future(price_paths[:, 1, :], portfolio_1[1]['strike'], r, np.arange(0, T+1) * 1/252)
price_paths[:, 1, :]

array([[10.        ,  7.58882033, 10.37962726, ..., -1.98002721,
        -3.41616904, -5.25086969],
       [10.        ,  9.67387548, 12.11915119, ...,  7.57796798,
         5.52481449,  9.52947136],
       [10.        , 10.36898839, 14.3633199 , ...,  6.95039868,
         3.14007929,  1.35431092],
       ...,
       [10.        , 11.18364291, 10.50876231, ..., 11.54770095,
        18.54570629, 16.80025319],
       [10.        ,  9.75640065, 11.97137626, ...,  6.2125986 ,
         6.9722392 ,  7.46943827],
       [10.        ,  9.54372245,  8.44735235, ..., 10.61552738,
         4.66027713,  0.4681688 ]])

In [67]:
price_paths

array([[[100.        ,  99.11413239, 100.29942991, ...,  92.25994578,
          92.04846301,  91.37875508],
        [ 10.        ,   7.58882033,  10.37962726, ...,  -1.98002721,
          -3.41616904,  -5.25086969]],

       [[100.        ,  97.41474872,  97.8352816 , ...,  98.65255726,
          97.85897518, 104.06664465],
        [ 10.        ,   9.67387548,  12.11915119, ...,   7.57796798,
           5.52481449,   9.52947136]],

       [[100.        , 101.47552889, 105.14435208, ..., 117.70502967,
         113.35788492, 116.07861471],
        [ 10.        ,  10.36898839,  14.3633199 , ...,   6.95039868,
           3.14007929,   1.35431092]],

       ...,

       [[100.        ,  99.20973941,  98.17244611, ...,  98.07906852,
          97.07990275, 100.62428786],
        [ 10.        ,  11.18364291,  10.50876231, ...,  11.54770095,
          18.54570629,  16.80025319]],

       [[100.        , 100.17216456, 100.78360242, ...,  94.94753438,
          94.26448554,  97.08250962],
       

In [68]:
# оптимизация, сразу определяем пси для всех сценариев
def portfolio_psi(portfolio, price_paths):
    quantities = np.array([i['quantity'] for i in portfolio]).reshape(len(portfolio), 1)
    ds = np.array([i['ds'] for i in portfolio]).reshape(len(portfolio), 1)

    mid = price_paths[:, :, 1:] - ds * price_paths[:, :, :-1]
    return mid * quantities


psi = portfolio_psi(portfolio_1, price_paths)
psi

array([[[  -8.85867612,   11.85297524,  -54.30377353, ...,
          -32.17315781,   -2.11482763,   -6.69707936],
        [ 241.11796688, -279.08069329,  481.47641718, ...,
          323.95277782,  143.61418353,  183.47006497]],

       [[ -25.85251277,    4.20532879,   20.58345237, ...,
          -14.91280645,   -7.93582079,   62.07669462],
        [  32.61245211, -244.52757134,  269.91525923, ...,
           28.90015694,  205.31534884, -400.4656867 ]],

       [[  14.75528886,   36.68823198,    7.20834874, ...,
           -3.87634469,  -43.47144752,   27.20729789],
        [ -36.8988391 , -399.43315126,   28.36638092, ...,
          358.24277229,  381.03193892,  178.57683651]],

       ...,

       [[  -7.90260586,  -10.37293307,   26.78184121, ...,
          -17.6474937 ,   -9.99165774,   35.44385109],
        [-118.36429064,   67.48805961,   42.06544297, ...,
         -122.14068626, -699.80053453,  174.54531033]],

       [[   1.7216456 ,    6.1143786 ,  -11.21277506, ...,
        

In [72]:
def get_optimal_strategy(portfolio, price_paths, T=10, param='WL'):
    assert portfolio is not None
    assert price_paths is not None

    quantities = np.array([i['quantity'] for i in portfolio]).reshape(len(portfolio), 1)
    ds = np.array([i['ds'] for i in portfolio]).reshape(len(portfolio), 1)
    abs_quantities = np.abs(quantities)

    if T == "auto":
        T = get_liquidation_period(portfolio)

    set_T = range(T)
    set_I = range(len(portfolio))

    import gurobipy as grb
    opt_model = grb.Model(name="MILP Model")

    # переменная ---> оптимальная стратегия
    opt_strategy = [
        [opt_model.addVar(vtype=grb.GRB.INTEGER,
                          lb=0, # ограничение 2
                          ub=abs(portfolio[i]['quantity']), # ограничение 3
                          name=f"Q_{i + 1}{t + 1}") for t in set_T]
        for i in set_I
    ]

    # ограничение 1
    opt_model.addConstrs(grb.quicksum(opt_strategy[i][t] for t in set_T) ==
                         abs(portfolio[i]['quantity']) for i in set_I)

    # ограничение 5
    opt_model.addConstrs(opt_strategy[i][t] == 0
                         for i in set_I
                         for t in range(portfolio[i]["execution_lag"] - 1))
    # ограничение 7
    opt_model.addConstrs(opt_strategy[i][t] <= portfolio[i]["daily_limit"]
                         for t in set_T
                         for i in set_I)

    # то ---> что оптимизируем
    L_var = opt_model.addVar(vtype=grb.GRB.CONTINUOUS,
                             lb=-grb.GRB.INFINITY)

    step1 = ds * (abs_quantities - np.cumsum(opt_strategy, axis=1)) +\
            opt_strategy
    step2 = psi * step1 / abs_quantities
    step3 = np.sum(step2, axis=1).cumsum(axis=1) # матрица L(r_k, h)

    first = step3[:, -1] # L(r_k, H)

    # ограничение 8
    if param.upper() == "WL":
        opt_model.addConstrs(L_var <= step3[pp][t] + first[pp]
                             for t in set_T
                             for pp in range(len(step3)))

    elif param.upper() == "PL":
        opt_model.addConstrs(L_var <= first[pp]
                             for pp in range(len(step3)))

    elif param.upper() == "TL":
        opt_model.addConstrs(L_var <= step3[pp][t]
                             for t in set_T
                             for pp in range(len(step3)))

    # for maximization
    opt_model.ModelSense = grb.GRB.MAXIMIZE
    opt_model.setObjective(L_var)
    opt_model.optimize()

    # L_var
    obj_val = opt_model.objVal
    # optimal strategy
    ans = np.array([[int(str(opt_strategy[i][t]).split()[-1][:-4]) for t in set_T] for i in set_I])

    return ans, obj_val

In [78]:
get_optimal_strategy(portfolio_1, price_paths, T=T, param='WL')

Gurobi Optimizer version 9.5.1 build v9.5.1rc2 (mac64[arm])
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 10031 rows, 21 columns and 190049 nonzeros
Model fingerprint: 0x2822e58f
Variable types: 1 continuous, 20 integer (0 binary)
Coefficient statistics:
  Matrix range     [3e-05, 7e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+01, 1e+02]
  RHS range        [6e-01, 1e+04]
Found heuristic solution: objective -6471.676213
Presolve removed 10031 rows and 21 columns
Presolve time: 0.02s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.03 seconds (0.02 work units)
Thread count was 1 (of 8 available processors)

Solution count 1: -6471.68 
No other solutions better than -6471.68

Optimal solution found (tolerance 1.00e-04)
Best objective -6.471676213264e+03, best bound -6.471676213264e+03, gap 0.0000%


(array([[  0,   0,   0,   0,   0,   0,   0,   0,   0,  10],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0, 100]]),
 -6471.676213264465)